# Hybrid Co-Simulation
--------------------------------

## Determinate Composition of FMUs for Co-Simulation

*Reference:*

Broman, David & Brooks, Christopher & Greenberg, Lev & Lee, Edward & Masin, Michael & Tripakis, Stavros & Wetter, Michael. (2013). Determinate composition of FMUs for co-simulation. 2013 Proceedings of the International Conference on Embedded Software, EMSOFT 2013. 1-12. 10.1109/EMSOFT.2013.6658580. 

### **1. Introduction**
- Goal: Achive deterministic execution of FMUs under the FMI standard for co-simulation (given the same input, the same output is produced every time)
- Maximal progress: the MA takes as large steps as possible without violating any constraints
- Problem with FMI 2.0 standard: FMUs lack I/O dependency information (direct feedthrough) and do not support rollbacks (saving and restoring internal states)
- Without these features, discrete event simulation and variable step size numerical integration is not possible
- Proposal: extension to the FMI standard that enables a master algorithm to query an FMU for the time of events that are expected in the future
- Requirements:
    - Model are either memoryless or implement one of rollback or step-size predictuon
    - Such a model can contain at most one 'legacy' FMU (without the proposed extension) that is not memoryless and does not implement rollback or step-size prediction

### **2. FMI for Co-Simulation**

- `fmiGetFMUState` and `fmiSetFMUState`: functions to save and restore the internal state of an FMU, providing rollback capability

```c
fmiStatus fmiDoStep(fmiComponent c,
                    fmiReal currentCommunicationPoint,
                    fmiReal communicationStepSize,
                    fmiBoolean noSetFMUStatePriorToCurrentPoint)
```
- By setting `noSetFMUStatePriorToCurrentPoint` to `fmiTrue`, the master *promises* that it will not call `fmiSetFMUState` for time instants prior to `currentCommunicationPoint`
- Still not entirely satisfactory for deterministic co-simulation, as the master cannot predict when the next event will occur within an FMU 
- Implementation of rollback procedures by and FMI is optional and may be difficult to achieve
- Saving and restoring state may be expensive in terms of memory and computation time
- FMI 2.0 interface does not bridge the gap between purely continuous-time and discrete-event simulation
- Time advancement in co-simulation $t$ to $t + h$ is done in fixed-size steps of size $h$
- Input values at time $t$ are provided to the FMU, and output values at time $t + h$ are obtained from the FMU
- Implicit solver: input value at $t+h$ will also be needed in order to determine the output at $t+h$
- FMI allows the MA to provide derivatives of the inputs at time $t$ to the FMU, FMU will use these derivatives to extrapolate input values within the interval $[t, t+h]$

### **3. Requirements of FMUs**
**Dependency Information**
- Question: In which order should the FMUs within a co-simulation scenario be executed?
- Helpful to know which outputs of an FMU depend immediately (direct feedthrough) on which inputs
- FMI provides such information in the Model structure of the model description XML file
- Without I/O dependency information, the MA needs information about the possible algebraic loops in the co-simulation scenario (solution depends on the initial guess for iterative solvers)
- Producers are called exacrtly once per communication step if there are no algebraic loops
- Idea can be generalized, constructing a topological sort of the involved ports in the model
- Internal dependencies and external dependencies (connections in the system) define a directed graph whose nodes are the ports of the FMUs
- A topological sort of this graph defines an execution order for the FMUs
- Acyclicity of the graph: if the graph does not contain cycles, the graph can be used to derive a correct evaluation order of all ports involved in the co-simulation scenario

**The Need for Rollback**
- FMI 2.0 includes functions to save and restore the internal state of an FMU,
- **Problem: implementation of rollback procedures by an FMU is optional and not mandatory**
- Saving and restoring the state of an FMU can be used to implement a rollback mechanism
- FMU rollback is required if one FMU accepts the proposed communication step size, but another FMU rejects it
- If an FMU accepts a communication step size $h$, state of the FMU is advanced to time $t + h$
- If an FMU rejects a communication step size $h$, state of the FMU must remain at time $t$
- Third possibility: FMU manages to make partial progress and advances its state to time $t + h'$, where $h' < h$
- In this case, the MA must rollback the FMU to time $t$ and try again with a smaller step size
- Rolling back procedure:
    1. MA uses `fmiGetFMUState` to save the state of all FMUs
    2. MA calls `fmiDoStep` on all FMUs with the proposed step size $h$
    3. If all FMUs accept the step size, proceed to the next communication point $t + h$
    4. If any FMU rejects the step size, MA uses `fmiSetFMUState` to restore the state of all FMUs to time $t$ and try again with a smaller step size
- It may appear that if any FMU rejects a step size, then all FMUs in the model must be rolled back

### **4. FMI Formalization**
**Core sets and notation**
- $C$: set of FMU instances in a model.
- $c \in C$: FMU instance identifier.
- $S_c$: set of states of instance $c$.
- $U_c$: set of input port variables of $c$.
- $Y_c$: set of output port variables of $c$.
- $\mathbb{V}$: universe of possible values.
- $D_c \subseteq U_c \times Y_c$: direct I/O dependency of $c$.
- Global inputs / outputs / dependencies:
  - $U = \bigcup_{c \in C} U_c$
  - $Y = \bigcup_{c \in C} Y_c$
  - $D = \bigcup_{c \in C} D_c$
- **Port mapping (connections)**: $P : U \rightarrow Y$
  Each input is connected to exactly one output.

**Formal FMU Interface**
- Initialization: $$init_c : \mathbb{R}_{\ge 0} \rightarrow S_c$$
- Set input: 
  $$set\_input_c : S_c \times (U_c \rightarrow \mathbb{V}) \rightarrow S_c$$
- Get output:
  $$ get_c : S_c \times Y_c \rightarrow \mathbb{V}$$
- Time advance:
  $$ doStep_c : S_c \times \mathbb{R}_{\ge 0} \rightarrow S_c \times \mathbb{R}_{\ge 0}  $$

**FMU contract – expected properties**
- Constraints are not always explicit in the FMI standard and some of them may not even be implicitly assumed
- Properties are crucial in proving the determinancy properties of the Master Algorithm
- Following guarantees that every FMU must provide and assumptions that every FMU instance makes

1. Signature of $doStep_c$ implies that a caller is not allowed to call it with $h<0$
2. $get_c(s, y)$ can only be called when $y \in Y$ (i.e., variable $y$ is indeed an output variable of $c$)

*Constraints*

- **A0:** if $doStep_c(s, h) = \left( s^\prime, h^\prime \right)$ then $0 \le h^\prime \le h$ (monotone step result)

- **A1:** if $doStep_c(s, h) = \left( s^\prime, h^\prime \right)$, then for any $h^{\prime\prime}$ where $0 \le h^{\prime\prime} \le h^\prime$, it holds that $doStep_c(s, h^{\prime\prime}) = \left( s^{\prime\prime}, h^{\prime\prime} \right)$ for some $s^{\prime\prime}$ (consistent step result)

- **A2:** Let $s^\prime = set_c(s, u, v)$. Then $s^\prime = s[u:=v]$

- **A3:** Let $v = get_c(s, y)$ and $v^\prime = get_c(s^\prime, y)$. If $s^\prime = s[u_1:=v_1, \ldots, u_k=v_k]$, and output variable $y$ does not directly depend on any of the input variables $u_1, \ldots, u_k$ (i.e., $(u_i, y) \notin D_c$ for all $i=1,\ldots,k$), then $v = v^\prime$ (non-direct dependency).



### **5. Determinate Execution**

#### **5.1 Algorithm Requirring Rollback**
*Algorithm 1: Order Variables*
- Creates a list of variables, describing the order that the variables may be acessed
- Uses the port mappeing $P$, global dependency relation $D$, and the set of variables $X$
- Creates a directed graph $G$ where the vertices are represented by port variables $X$ and an edge is a variable dependency or a connection
- A topological sort of $G$ is performed to obtain an ordered list of variables. If a cycle is found, the algorithm fails.

*Algorithm 2: Master Step*
- Input: Set of instances $C$, ordered list of variables $\bar x$, port mapping $P$, maximal step size $h_{max}$, and a mutable state mapping $m$ of size $|C|$ where $m[c]$ is the current state of instance $c$
- Output: Updated state mapping $m$ and the performed step size $h$
1. Set values for all input variables in order $\bar x$
2. Save the states of all FMUs to enable rollback if needed
3. Set communication step size to an initial default value $h := h_{max}$
4. Find $h$ acceptable by all FMUs:
    - For each FMU $c \in C$, call $doStep_c(m[c], h)$ \
    - If all FMUs accept $h$, update their states and return
    - If any FMU rejects $h$, rollback all FMUs to their saved states and reduce $h$ accordingly
5. Repeat step 4 until an acceptable $h$ is found or $h$ becomes zero
6. Return updated state mapping $m$ and step size $h$

*Theorem 1: Determinany of Algorithm 2*
- Algorithm 2 is determinate, i.e., for given inputs $C$, $P$, $h_{max}$, and $m$, it always produces the same output state mapping $m$ and step size $h$.
- Does not depend on the order of FMU execution or the order of variable assignments within an FMU
- Correct in the sense that if $m_2$ is the state mapping after executing Step 1, and $m, h$ are the values returned by algorithm 2, then for all $c \in C$, it holds that $doStep_c(m_2[c], h) = (m[c], h)$.
- Algorithm 2 returns an acceptable step size $h$

*Theorem 2: Maximal Progress of Algorithm 2*
- Let $m_2$ be the state mapping after executing Step 1 of Algorithm 2.
- Let $h$ be the step returned by Algorithm 2.
- There is no step size $h'$ such that $h<h^\prime \le h_{max}$ and $h^\prime$ is acceptable at $m_2$



#### **5.2 Predicatble Step Sizes**
- Rollback may be impractical, particularly when an FMU wraps legacy coder or serves as a wrapper for a simulation tool7
- With small addition to the FMI standard, such FMUs can be handles in certain cases
- Proposed extension:

```c
fmiStatus fmiGetMaxStepSize(fmiComponent c,
                            fmiReal *maxStepSize)
```

- Argument `maxStepSize` returns an upper bound on the step size that the FMU can accept (infinity if there is none)
- Can be zero to inddicate the need for a zero step sizes step
$$ \text{getMaxStepSize}_c : S_c \rightarrow \mathbb{R}_{\ge 0} \cup \{\infty\} $$
- Let $C_P$ be the set of FMU instances that implement this function
- We have the following requirement for such FMUs:
- **A4**: If $c \in C_P$ and $s \in S_c$ and $\text{getMaxStepSize}_c(s) = h $ then for all $h^\prime$ where $0 \le h^\prime \le h$, it holds that $doStep_c(s, h^\prime) = (s^{\prime}, h^\prime)$ for some $s^{\prime}$ (predictable step size)
    - Means that an instance will accept any time step smaller than or equal to the time step returned by $\text{getMaxStepSize}$
    - Capability should be indicated in the model description XML file of the FMU
- Legacy FMUs do not implement this function and are not part of $C_P$
- A model that composes FMUs has at most one legacy FMU instance, and the remaining FMU instances are part of $C_P$

*Algorithm 3: Master-Step with Predicatble Step Sizes*
- Input: Set of instances $C$, ordered list of variables $\bar x$, port mapping $P$, maximal step size $h_{max}$, and a mutable state mapping $m$ of size $|C|$ where $m[c]$ is the current state of instance $c$
- Output: Updated state mapping $m$ and the performed step size $h$
1. Set values for all input variables in order $\bar x$
2. Find the minimal predicatble communication size: \
    $h := \min_{c \in C_P} \text{getMaxStepSize}_c(m[c])$
3. Save the states for all instances that can perform rollback
    1. For each FMU $c \in C$ that supports rollback, save its state
    2. `doStepOnLegacy := true`
    3. Goto step 5
4. Restore states for rollback instances
5. Perform `doStep` on all instances with rollback: \
    $h_{min} := h$\
    For each FMU $c \in C_R$:
    1. $(s^\prime, h^\prime) := doStep_c(m[c], h)$
    2. $h_{min} := \min(h^\prime, h_{min})$
    3. $m[c] := s^\prime$
    - After this loop, all rollback FMUs can accept exactly step size $h$
6. If $h_{min} < h$ then $h := h_{min}$ and goto step 4
7. Perform `doStep` on legacy instance (if any): \
    If $c \in C_L$ and `doStepOnLegacy` is true:
    1. $(s^\prime, h^\prime) := doStep_c(m[c], h)$
    2. $m[c] := s^\prime$
    3. doStepOnLegacy := false
    4. If $h^\prime < h$ then $h := h^\prime$ and goto step 4
    - Ensures that the legacy FMU can also accept step size $h$
8. Perform `doStep` on all FMUs with predicatvle step sizes: \
    For each FMU $c \in C_P$:
    1. $(s^\prime, h^\prime) := doStep_c(m[c], h)$
    2. Assert $h^\prime = h$
    3. $m[c] := s^\prime$
9. Return updated state mapping $m$ and step size $h$

Result:
- All FMUs have been advanced by the **same, globally valid step size `h`**.
- `h` is the **maximal** step that respects:
  - predictions from `C_P`,
  - actual capabilities of `C_R` and the single `C_L`.

*Theorem 3: Determinany of Algorithm 3*
- Algorithm 3 is determinate, i.e., for given inputs $C$, $P$, $h_{max}$, and $m$, it always produces the same output state mapping $m$ and step size $h$.
- Does not depend on the order of FMU execution or the order of variable assignments within an FMU



--------

## Setp Revision in Hybrid Co-Simulation with FMI

*Reference:*

F. Cremona, M. Lohstroh, D. Broman, M. Di Natale, E. A. Lee and S. Tripakis, "Step revision in hybrid Co-simulation with FMI," 2016 ACM/IEEE International Conference on Formal Methods and Models for System Design (MEMOCODE), Kanpur, India, 2016, pp. 173-183, doi: 10.1109/MEMCOD.2016.7797762.

### **1. Introduction**

**Cyber-Physical Systems (CPS)**
- Integration of many components including physical parts (mechanical, electrical) and cyber parts (software, communication).
- Each component is modeled using its own modeling formalism and simulation tool.
    - State machines for software behavior.
    - Differential equations for physical dynamics.
- Modeling and simulation is a non-trivial task
- UML and SysML: standard modeling language for software systems
- Simulink: standard tool for signal processing
- Modelica: standard equation-based language for physical systems

**Co-Simulation**
- seeks to levearge existing modeling and simulation tools by coupling them together.
- Provide capability to simulate complex CPS by integrating different models and simulation tools.
- Each component is simulated by its own simulator.
- Connect and co-simulate sub models by means of an integration and coordination framework.

**Functional Mock-up Interface (FMI)**
- International standard for model exchange and co-simulation of dynamic systems.
- FMI defines a standard application programming interface (API) for simulation tools to communicate and exchange data.
- Component in FMI is called Functional Mock-up Unit (FMU).
- For co-simulation, an FMU is a 'black box' with input ports, output ports, and a set of state variables
- FMU implements functions in the API to initialize state variables, set the value of input ports, get the value of output ports, and advance the simulation time.
- FMU can internally implement a simulator that is different from the simulator used in other FMUs.
- FMI standard does not specify a **master algorithm (MA)** which orchestrates the co-simulation process.

**Master Algorithm (MA)**
- Orchestrates the co-simulation process by coordinating the execution of multiple FMUs.
- Responsible for:
    - Initializing FMUs
    - Advancing simulation time
    - Exchanging data between FMUs at each time step
- Has to ensure determinism and consensus on the size of each time step among all FMUs.
- Key feature is **step *determination**, which entails finding the largest step that is acceptable for all FMUs involved in the co-simulation.

**Zero-crossing detector (ZCD)**:
 - Component that takes as an input a continuous-time, real-values signal $x(t)$ and outputs a discrete event evry time $x(t)$ crosses zero.
- Suppose $x(t) = -1$ and the master advances to $t + h$.
- To know whether a zero-crossing has occured, the ZCD also needs to evaluate $x(t + h)$. This value is not provided in the FMI API until *after* the step has been accepted and commited by all FMUS.
- Thus, ZCD does not know whether a zero-crossing has occured and need to accept the step $h$ in order to find out if a zero-crossing has occured.
- Suppose that the input to the ZCD is $x(t + h) = 1$. The ZCD now knows that a zero-crossing has occured within the interval $[t, t + h]$.
- *Question*: Should the ZCD generate an output event at the current time $t + h$?
- *Answer*: Depends on the error parameters with which the ZCD is configured.
- Generating an output event at time $t + h$ is unacceptable, if the *uncertainity interval* (the interval within which the zero-crossing occured) is larger than the configured error tolerance.
- In that case, the entire simulation model needs to be rolled back to time $t$ and the step size $h$ needs to be reduced (**step revision**)

### **2. Simulation of Hybrid Systems**

#### A. Principles of Numerical Integration

$$ x(t) = x_0 + \int_{t_0}^{t} \dot x(\tau) d\tau $$

- $\dot x$: time derivative of the integrated function
- $x_0$: initial value at time $t_0$
- $t_0$: initial time from which integration starts

*Ordinary Differential Equations (ODEs)*
- $\dot x(t)$ at any time is a function of the current state $x(t)$, the current input $u(t)$, and time $t$ itself:
$$ \dot x(t) = f(x(t), u(t), t) $$
- If $f$ is analytic, the the value of $x(t+h)$ at some future point is given by the Taylor series expansion:
$$ x(t + h) = x(t) + h \dot x(t) + \frac{h^2}{2} \ddot x(t) + \ldots $$
- Converges fast to the true value of $x(t+h)$ as the terms in the series become small
- **ODE solvers** rely on the truncation of the Taylor series to estimate the new value $x(t+h)$
- **Variable-step size technques**: adapt the step size $h$ dynamically during simulation to achieve a desired accuracy
- **Fixed-step size techniques**: use a constant step size $h$ throughout the simulation
- **Runge-Kutta methods**: explicit RK solver uses a variable step-size and the evaluation of $f(x(t), u(t), t)$ at several points within the interval $[t, t+h]$ to compute $x(t+h)$
- **Loacal truncation error**: incurred by a single integration step from $t$ to $t+h$ is of order $O(h^{k+1})$ where $k$ is the order of the RK method
- In **FMI** the evaluation of an FMU's output and the computation of its new state are performed inside the same API function `fmiDoStep`
    - Input values at time $t$ are provided to the FMU before calling `fmiDoStep`
    - During the integration from $t$ to $t+h$, the FMU internally extrapolates the input values within the interval $[t, t+h]$
    - Hence it can only guess whether an zero-crossing has occurred 

#### B. Models of Computation
- Challenge in co-simulation is the orchestration of the interactions between different concurrently operating components each with its own simulator
- Requires coordination rules that the simulation follows, and which make components synchronize a certain way and exchange data at certain points in time between them
- FMI: components are called FMUs and the orchestrator is called the master algorithm (MA)
- **Hybrid Simulation**:
    - Combines both continuous-time and discrete-event simulation
    - Discrete event is only present at a precise time instant and is absent otherwise
    - Discontinuity is an instantaneous chanhe in s continuous-time signal

#### C. Continuous-Time Simulation

*Simulink*
- S-function is executed in a split phase manner
- Divides further into a **major step** and several **minor steps**
- Major step: simulation engine executes the functions for the output update and state update (exactly once per simulation time step)
- Minor steps: Integration process executes in minor steps where the solver computes the outputs and then updates the states (several times per simulation time step)
- Simulink performs at each minor step a check for zero-crossings in the continuous-time signals
    - First computes the outputs
    - Then checks whether a zero-crossing has occured
    - Once deteceted, solvers uses a bisection to determine the point in time when the zero crossing occured

*FMI*
- FMI does not provide a two phase API, hence `fmi2DoStep` **is not free of side effects**
- Inputs of an FMU are supplied before each integration step, thus a third-order RK algorithm embedded in an FMU can only use extrapolation to estimate the inputs at $t, t+0.5h, t+0.75h$
- Extrapolation yields an approximation of the values of inputs past $t$, this extrapolation error wull factor into the integration error
- Alternative: Accept time advancement to $t+h$ to evaluate inputs at $t+h$ and then perfrom integration using **interpolation of the inputs** within $[t, t+h]$
    - If the integration error is too large, the **step needs to be revised** and the step size reduced

#### D. Combining Continuous-Time with Discrete Events
- **Time Events**: occur at specific points, may be predicted and time is known
- **State Events**: are not known in advance, require the evaluation of the state in order to be detected
- *Zero-Crossing function = guard function*:
$$g(x(t), x(t+h))$$
- Determines when the level crossing occurs by evaluationg *true* when:
$$sign(x(t)) \neq sign(x(t+h))$$
- When $g(x(t), x(t+h))$ evaluates to true, an event is generated at time $t+h$
- Detection of a zero-crossing does not actually reveal the exact time when the crossing occurred due to finite precision of arithmetic in digital computers
- **Uncertainty interval**: interval within which the zero-crossing occurred
- If the uncertainty interval is larger than a configured error tolerance, the step needs to be revised
- The occurence of a zero-crossing can only evaluated after the input to the ZCD at time $t+h$ has been obtained
- Consequences: Upstream components must be evaluated at time $t+h$ and then might have to roll back to time $t$ if the step is revised
- Bisection Search: Techniques to revise the step size in order to more accurately identify the time at which a zero crossing occurred
- FMI: evaluation of signals is not free of side effects, step size revision involves restoring each component to its last known valid staten and the re-evaluating all outputs at the time of the event

*Requirements for the Extrapolation Strategy*
1. The MA must be able to propagate derivatives provided by the FMUs to enable extrapolation of inputs within a communication step.
2. Each FMU in the path from a signal source to the components the components that want to extrapolate its input signals must be able to propagate the derivatives.

- Propagtation of derivatives to the ZCD requires an *Adder* so that it accepts an input derivative and also performs the add opeartion on the value of that derivative before propagtaing it
- Problem: support fopr derivatives is optional in the FMI standard
- Step size of the ZCD:
    - Very small: simulation will become inefficient
    - Very large: detector may overshoot zero-crossings due to oversampling
- Proposal: MA that supports step size revision triggered by state events
    - Allows the ZCD to rely solely on the simple guard function $g(x(t), x(t+h))$ without the need for derivatives
    - Such approach is more composable since it does not depend on the presence of derivative propagation in the FMUs

|Time events|State events|
|-----------|------------|
|![time_event](figures/hybrid/time_events.png)|![state_event](figures/hybrid/state_events.png)|

### **3. Step Determination and Step Refinement**

- ZCD can oonly detect the sign switch and evaluate the error **after** having commmited to a step size and having received new inputs
- Ovbershooting and exceeding of error tolrances requires an **undo** of the last step and a **redo** with a smaller step size

*Step Determination*
- Algorithm supposes that some or none FMUs can report their maximum step size (finds the minimimum among these maximim step sizes, defines the initial step size $h$)
- Algorithm lets the other FMUs perform a step with size $h$
- FMU can accept and reject the step size by returning different codes from the `fmiDoStep` function:
- All speculatively executed FMUs (for step determination) must be rolled back to their previous state, then all FMUs can be moved forward be the determined step

*Step Revision*
- Involves moving the entire system back in time befor moving forward again with a smaller step size
- Necessary in FMI in order to support co-simulation of FMUs that implement numerical approximation algorithms that suffer an error that is dependent on the step size (R-K solvers and ZCD)
- FMU can indicate the need for step revision by returning a specific code from the `fmiDoStep` function
1. Save the state of all FMUs at the current time $t_0$, determine $h$, and attempt to advance to $t_2 = t_0 + h$
2. If any FMU returned an invalid state for getMaxStepSize or doStep returned an error, restore all FMUs to their saved states at time $t_0$, reduce to $h^\prime$, and attempt to advance to $t_1 = t_0 + h^\prime$ again
3. If all FMUs accepted the step, repeat step 1
4. Advance time to $t_1$ by calling doStep on all FMUs with step size $h^\prime$
5. If all FMUs arrive in valid state, then go to 1 and repeat the above sequences with $t_1$ as the current time, otherwise repeat with yet a smaller $h^\prime$
- Step revision is complementary to step determination
- After a step has been determined (acceptable by all FMUs), some may still end up in an invalid state requiring step revision
- Step revision: speculation reaches after the speculative step has concluded
- Step determination is guaranteed to terminate in one iteration
- Step revision may require several ietartions. It is not recursive, it will only go back and forth between $t+h$ and $t$, each time reducing the step size $h$

![Step Determination and Step Revision](figures/hybrid/step_determination_revision.png)

#### A. Support for Rollback in FMI
- FMI 2.0 specifies functions `fmiGetFMUState` and `fmiSetFMUState` to save and restore the internal state of an FMU (but the implementation is optional)
- Suggestion to make the implementation of these methods mandatory in order to enable support for components that use state events to control the simulation step size and if needed request step size revision
- Proposal: More reasonable to rely on rollback abilities than it is to rely on the extrapolation techniques that uses derivatives and hence require FMU to be able to work with derivatives, even if they don't need them internally
- **Requirement**: All participating FMUs must implement rollback via `fmiGetFMUState` and `fmiSetFMUState`

### **4. Extensions to the FMI Standard**
- FMI-CS 2.0 is optimized for simulating of continuous dynamcis but lacks support for discrete-event simulation and handling discontinuities

#### A. Hybrid Co-Simulation
1. FMU should be able to indicate its output or detect its input to be *absent*
2. It must support zero-duration steps in order to support discrete discontinuities

- The following aspects of FMI-CS 2.0 are incompatbile with these requirements:
   1. The step size proposed to an FMU with `fmiDoStep` must be strictly grater than zero
   2. The type system of FMI does not support absent signals. Signals are defined in a time-continuum, they are never absent.

- **Proposal**:
- For absent signals: augment the FMI functions for getting and setting values to indicate wheter a signal is present or absent
- Zero-duration events or discontinuities: can be achieved by exetnding the standard with the superdense model of time
    - Time is represented as a tuple $\tau = (t, n)$ with $t \in \mathbb{R}_{\ge 0}$ and $n \in \mathbb{N}_0$
    - First element $t$ represents the physical time (Newtonian time)
    - $n \in \mathbb{N}_0$ is the index of the current iteration at the same Newtonian time $t$
    - Can be supported by simply enabling the FMU to accept a step size $h \ge 0$ in `fmiDoStep`
    - Interpret input changes at a superdense time index greater than zero as discrete changes
    - **Discontinuity**: present value at $(t, 0)$ and a present value at $(t, 1)$
    - **Discrete event**: absent value at $(t, n)$ and a present value at $(t, n+1)$, and absent value at $(t, n+2)$

#### B. Algebraic loops
- **Algebraic loop** occurs when an FMU with direct feed through is placed in a feedback loop
- **Strict FMU**: one or more outputs that depend on one or more inputs at the same time instant
- When a zero-delay path is created between such output and one of the inputs that it depends on, **direct feedthrough** occurs, and the assumed causal relationship between FMU's input and output is broken
- Not all loops are algebraic loops
- **Artifical Algebraic Loop**: loop that contains strict components may be false identified as an algebaric loop in case the strict components in fact do not allow direct feedthrough between the particular ports that connect them in a loop
- Causality can be restored by inserting a non-strict component in an algebaric loop (delay between input and output), but this may not always be possible or desirable
- Solving algebraic loops requires smooth functions and are incompatible with discrete changes
- Particularly in the case of hybrid co-simulation, algebraic loops should be rejected
- **Melay Machine**:
    - Requires separate functions for computing outputs and transitioning between states
    - FMI prescribes that both output computation state updates are performed in the same function `fmiDoStep`
- **Proposal**:
    - Perform output computation in `fmiGetXXX` functions
    - State update in `fmiDoStep`

#### C. Predictable Step Sizes
- Ability of an FMU to communicate to the master the size of the maximum step it is able to take
- Requires a function called `fmi2GetMaxStepSize`
- Reason: increase the efficiency of the algorithm
- Saving and restoring state of the FMU is an expensive operation (time and memory)
- Prevents the master from taking small steps if non of the participating FMUs have an interest in taking such small steps



### **5. Master Algorithm for Hybrid Co-Simulation with Step Revision**

#### A. A Formalization of the FMI Standard
- Each FMU $c \in C$ is a black-box model with input port variables $U_c$, output port variables $Y_c$, and a set of state variables $S_c$
- Each variable type defined in the FMI standard is extended to include the absent value
    - $\mathbb{V}_\epsilon$: union type that ranges over all possible data types in FMI, augmented with a value for absent (Example: `fmi2Real`)
- $\mathbb{M}$: set of all possible values for the flags returned by a function call `fmi2DoStep`or `fmi2GetMaxStepSize` (e.g., OK, Discard, and Error)
    - return OK: step accepted or step size is valid
    - return Discard: FMU cannot perform an entire step but completes a fraction of it
    - return Error: FMU cannot perform the step due its current state or input values

*Functions*
- $\text{init}_c : \mathbb{R}_{\ge 0} \rightarrow S_c$
    - performs a sequence of opertaions to initialize the FMU $c$
    - Includes `fmi2SetupExperiment`, `fmi2GetXXXHybrid`, `fmi2SetXXXHybrid`, to determine initial conditions and retrieve parameters
- $\text{set\_input}_c : S_c \times U_v \times \mathbb{V}_\epsilon \rightarrow S_c$
    - $\text{set}_c(s, u, v)$ assigns the value $v \in \mathbb{V}_\epsilon$ to the input port variable $u$ of FMU instance $c$ and returns the updated state $s\in S_c$
    - call updates the inputs of the FMU
- $\text{get}_c : S_c \times Y_v \rightarrow \mathbb{V}_\epsilon$
    - $\text{get}_c(s, y)$ retrieves the value $v \in \mathbb{V}_\epsilon$ of the output port variable $y$ of FMU instance $c$ 
    - call affects the value of an output variable based on the FMU's current state and inputs
    - multiple calls to `get` with the same state and same input lead to the same output value, call does not change the state of the FMU
- $\text{doStep}_c : S_c \times \mathbb{R}_{\ge 0} \rightarrow S_c \times \mathbb{R}_{\ge 0} \times \mathbb{M}$
    - Performs a state update on the FMU
    - Call returns the tuple $(s^\prime, h^\prime, m)$
    - $s^\prime$: new state of the FMU after performing the step
    - $h^\prime$: the performed step size
    - $m$: status flag indicating whether the step was accepted, partially accepted, or rejected
    - By having it take the current state as an input argument and letting it return the new state obtained by completing a step, we can use it to model rollback without introducing extra primitives for retrieving and restoring the state
- $\text{getMaxStepSize}_c : S_c \rightarrow \mathbb{R}_{\ge 0} \times \mathbb{M} $
    - Returns a tuple $(h, m)$ where $h$ is the maximum step size and $m$ is a status flag indicating whether the returned step size is valid or not
    - Function has no side effects and does not change the state of the FMU

*Assumptions*
- Each FMU is modeled as a Mealy machine with some internal I/O dependencies $D_c$
- If a model contains an algebraic loop, then the model is discarded
- The graph $\mathbb{X}$ corrsponds to a valid model (acyclic)
- Vertices are port variables $U \cup Y$
- Edges are $E = D \cup \{(y, u) | u \in U \text{ and } P(u) = y\}$
- Graph can be totally ordered using Kahn's algorithm for example
- Totally ordered acyclic graph $\bar{\mathbb{X}}$ describes the order in which variables can be accessed without violating dependencies

#### B. The Algorithm

**Algorithm 1:**
- Orchestrates the co-simulation (set of FMUs $C$) from $t_{start}$ to $t_{end}$
- Algorithms tries to advance at each simulation time step from a time instant $t$ to $t + h$ where $h$ is the computed step size
- Algorithm can rollback each FMU to time $t$ if any FMU rejects the proposed step size $h$ or returns an error from the invocation of `doStep`
- Vectors $r$ and $r^\prime$ store all states during the execution of the algorithms
- Vector $s$ stores the current states of all FMUs
- **Rollback:**
    1. $r$ stores the state after the last I/O port update prior to a speculative step
    2. Collective state after the last successful execution is stored in $r^\prime$

*Four Phases of a Simulation*
1. **Initialization**:
    - Master iterates over all FMUs to setup the simulation times
    - Master sets the initial value of variables and parameters
2. **Input / Output Update**:
    - Master iterates over the totally ordered set of ports
    - For each input port $u$, function $P$ returns the output port $y$ which is driving $u$
    - Value of the output port $y$ is obtained and utilized to update the respective input port $u$
    - All states are saved in $r$ and `errorState=false`after all input ports have been updated (flag indicates whether rollback is needed)
3. **Step Determination**:
    - Master determines the maximum step size $h$ that is acceptable by all FMUs
    - Predictable FMUs
        -  ($C_P \subseteq C$) report their maximum step size using `getMaxStepSize`
        - accept any step size smaller than or equal to the reported maximum step size
        - Partial progress is indicated by returning the `Discard` flag from `doStep`
        - Acceptance of the step is indicated by returning the `OK` flag from `doStep`
        - Error state is indicated by returning the `Error` flag from `doStep`
    - Unpredicatble FMUs ($C_U = C \setminus C_P$) which do not implement `getMaxStepSize` are speculatively executed using `doStep`
    1. Setting a default value $h = h_{max}$
    2. Iterate over the predictable FMUs to find the minimum maximum step size and find the minimum among these values is the maximum step size that can be accepted by all predictable FMUs
    3. Iterate over the unpredictable FMUs and test whether they accept a step size of $h$, FMUs that reject will report partial progress
    4. Next simulation step size is the minimum among all reported partial progress step sizes and the computed step size $h$
    5. Predictable FMUs are unsynchronized and must be rolled back to their saved states in $r$
    6. If any unpredictable FMU reported an error state, set `errorState=true`, and the master restores all FMUs to their saved states in $r$
4. **Step Revision**:
    - If the error state flag is set to true, the model must be rolled back to the last successful state stored in $r^\prime$
    - Current state $s$ is restored to $s := r^\prime$
    - New step size $0 \leq h < h_{prev}$ is chosen
    - Master backtracks the simulation time to $t := t - h_{prev}$ and saves the computed step size $h_{prev} := h$ (case where also the next iteration yields an error flag)
5. **Step update**:
    - If `errorState=false`, the algorithm saves the initial state $r^\prime := r$ and performs a state update on all FMUs
    - Master saves the computed step size $h_{prev} := h$ and advances the simulation time $t := t + h$
    - Master resets the step size to the maximum allowed value $h := h_{max}$ for the next iteration
    - States $r^\prime$ and the step size $h_{prev}$ will be used to backtrack the simulation if needed

**Example: Bouncing Ball**
![bouncing_ball](figures/hybrid/bouncing_ball.png)

### **6. Conclusion**
- FMI-CS 2.0 standard is not suited to simulate hybrid systems
- Needs some extensions to support discrete events and zero-duration steps
- Proposed master algorithm that supports step determination and step revision in hybrid co-simulation
- Algorithm relies on the ability of FMUs to rollback to previous states
- Rollback requires functions `fmiGetFMUState` and `fmiSetFMUState` to be implemented by all participating FMUs
- Step revision allows FMU to interpolate continuous input signals instead of extrapolating them
- FMUs without extrapolation capabilities are simpler and more composable and introduce step-size dependent errors can be kept within bounds by the master that performs step revision

---

## Hybrid Co-Simulation: it's about time

*Reference:*

Cremona, Fabio & Lohstroh, Marten & Broman, David & Lee, Edward & Masin, Michael & Tripakis, Stavros. (2018). Hybrid Co-simulation: It's About Time. MODELS '18: Proceedings of the 21th ACM/IEEE International Conference on Model Driven Engineering Languages and Systems. 368-368. 10.1145/3239372.3242896.

### 1. Introduction

- FMI 2.0 is unable to simulate mixed discrete-continuous models in a satisfactory manner
- Hybrid co-simulation: loose copling of components within a co-simulation scenario but with support for discrete and discontinuos signals and instantaneous events
- Goal: Support of hybrid systems which combine continuous dynamics with discrete events (required for interactions between cyber and physical components)
- Central problem is the **modeling of time**: floating point representation of time in FMI 2.0 is insufficient to model discrete events and discontinuities
- Proposal using a form of **superdense time** allows events to occur discretely in time and in sequence without time advancing
- Goal: Support components that use floating-point time respresentation together with components that use superdense integer time representation

### 2. Representing Time

#### 2.1 Models of Time

**Order of Events**
- Notion of time has to provide a clear ordering of events. Each component in a system must be able to differ between past, present, and future.
- The *state* of a component at a *present* time contains all information about the *past*
- Component state changes as time advances to the *future*
- All observers should see state changes in the same order

**Causality**
- Semantic notion of time needs to respect causality
- If an event $A$ causes an event $B$, then $A$ must occur before $B$ in time
- The order must be observable for all observer

**Simultaneity**
- Two events are simulatneous, if all observers see them occurring at the same time
- Models where one observes deems two events to be simulateneous and another does not needs to be avoided

**Problem with Newtonian Time**
- Time represented as a single real-valued number $t \in \mathbb{R}_{\ge 0}$
- Floating-point representation of real numbers introduces precision errors
- Digital computers cannot represent all real numbers exactly due to finite precision
- Quantization errors accumulate over time
- Comparing for simulateneity becomes unreliable when comparing floating-point numbers

**Example**
- Model where two components (event sources) produces periodic events with the same period starting at the same time
- Model paradigm should assure that those events will appear simulatenously to any other component that observes them
- Without this notion, the order of these events will be arbitrarily
- Changing the order of events may have a bigger effect than perturbing samples of continuous signals
- Quantization erros shall not weaken simultaneity guarantees

**Requirements for a Time Model according to Broman et. al:**
1. The **precision** with which time is represented is finite and **should be the same for all observers in a model**. If the precision between observers differs, simultaneity cannot be guaranteed.
2. The **precision** with which time is represented should be **independent of the absolute magnitude of time**. The choice for meaning of time zero should not affect the precision.
3. Addition of time should be **associative**. The order in which time intervals are added should not affect the result.
$$(t_1 + t_2) + t_3 = t_1 + (t_2 + t_3)$$

**Time Resolution = Precision of Time Representation**

**Definition:** Time resolution is the smallest representable time difference betwee two time stamps.

- Example: Resolution of 1 millisecond means that two time stamps $t_1$ and $t_2$ are considered equal if $|t_1 - t_2| < 0.001$ seconds
- Time points between $t$ and $t + \text{resolution}$ are indistinguishable and cannot be represented separately
- Floating-point time reoresentation should not be used as the primary representation of time if there is a need for the notion of simultaneity


#### 2.2 Superdense Time
- Superdense time can be represented as a pair $(t, n)$ called a **time stamp**
    - $t$ model time represents the time at which some event occurs
    - $n$ microstep (index) represents the sequencing of events that occur at the same mode time
- Two time stamps $(t_1, n_1)$ and $(t_2, n_2)$ can be considered as **simulataneous** in a weak sense if $t_1 = t_2$
- Strong simultaneity requires both $t_1 = t_2$ and $n_1 = n_2$
- Time stamps can be ordered lexicographically:
$$(t_1, n_1) < (t_2, n_2) \text{ if } t_1 < t_2 \text{ or } (t_1 = t_2 \text{ and } n_1 < n_2)$$
- An event is considered to occur before another event if its time stamp is smaller according to the lexicographic order
- An **event** is value with a time stamp
- Since computers cannot prefectly represent real numbers, a time stamp of the form $(t, n) \in \mathbb{R} \times \mathbb{N}_0$ is not realizable in software
- Microstep can also be a problem when it grows unbounded

#### 2.3 Integer Time

- Integer values can be interpreted as representing a time value with some arbitrary unit (e.g., microseconds, nanoseconds, etc.)
- Integers can be added and subtracted without introducing rounding errors (required for enabling simulateneity)
- If floating point numbers are used to represent time, and a tolerance is defined for simulatenity, then when we have three events $A, B, C$, it is possible that $A$ and $B$ are simultaneous within the tolerance, and $B$ and $C$ are simultaneous within the tolerance, but $A$ and $C$ are not simultaneous within the tolerance (violates transitivity of simultaneity)
- Time resolution should be the **same for all observers** in a model (event listener)
- Time resolution needs not to be the same across different models / components
- Simulation models tend to have different time scales (e.g., mechanical systems vs. software systems)
- Co-Simulation: Each model can operate with its own time resolution
- Master algorihm perspective: Components must progress in increments that are multiples of the time resolution used by the master
- Claim: It is sufficent for the master to coordinate FMUs by adding, subtracting, and multiplicaty by integers only. Thus, an integer representatio of time can be very efficient.
- FMUs / components that use floating-pint representation of time require conversion of integer time to floating-point time and vice versa
- FMUs with floating-point time representation should never
    - Compare two time stamps for equality
    - Have no behavior that depends on strongly on the relative ordering of two time values
- **Selection**: represent time with a 64-bit unsigned integer with arbitrary resolution
    - Resolution is a parameter of the model and origin equal to the simulation start time
    - For well-chosen resolution, long simulation times can be represented without overflow
    - Conversion between integer time and floating-point time is possible with loss
    - Fixed universal resolution for all components does not make sense due to different time scales

#### 2.4 The choice of resolution
- FMU may be only valid over a range of time scales $\to$ the FMU should be able to insist on a resolution
- FMU should also be capable of adapting to a finer resolution then the one it requests
- **Challenge**: Two FMUs with different resolutions need to interact
- **Possibilities**:
    1. Selected resolution for the model is the finest of all specified resolutions
    2. Selected resolution for the model is the greatest common divisor (GCD) of all specified resolutions

**Possible Data Types**
- **Double**:
    - All components share a single double-precision floating-point number
    - Represents the unit shich specifies the resolution
    - Time stamps are interpreted as an integer multiple of this value

- **Rational**:
    - Resolution is specified as a fraction $\frac{p}{q}$ where $p, q \in \mathbb{N}$
    - This allows always to finda a GCD between two resolutions
    - There is a risk of overflow if the numerator and denominator are represented with a bounded number of bits
    - Conversion to and from floating-point time may be more expensive

- **Decimal**:
    - Resolution is specified using an integer exponent $n$ that stipulates a resolution of $10^{n}$
    - Allows that the fines resolution is always the same as the GCD and is always precise
    - Allows that any parameter specified as decimal numbers can be represented exactly (as long as the resolution is fine enough)
    - Adavantage: Calculation of differences between two parameter values is without quantization error

- Not every FMU may be required to adapt to a finer resolution than the preferred resolution it declares (e.g., if an FMU generates events exculisevly only at multiples of its preferred resolution)

### 3. Hybrid Co-Simulation with Integer Time
- Extends the FMI-CS 2.0 standard by
    1. Encoding the absence of an event
    2. Allow an FMU to react instantaneously without moving forward in time (zero-duration steps)

#### 3.1 Extension to the FMI Standard
- FMUs and Master Algorithm (MA) need to agree on a common time resolution before the simulation starts
    - `getPreferredResolution(fmiComponent c, fmiTimeResolutionExponent *n )`
    - `setResolution(fmiComponent c, fmiTimeResolutionExponent n )`
- Hybrid Step function `fmi2DoHybridStep` that uses integer time representation instead of floating-point time
- `getMaxStepSize`: returns the maximum allowed communication step size
- `getHybrid` and `setHybrid`: get and set input and output variables that can be absent to indicate that there is no value present at the current time

#### 3.2 Categories of FMUS in FMI-HC

**Category 0:**
- FMU uses intenally floating-point time representation
- *Category 0a*: not suitable for hybrid co-simulation because the standard disallows zero-step sizes and insists on calling `doStep`in between every input / output update
- *Category 0b*: Allow zero-step size and getting and setting values (multiple times) without having to advance time

**Category 1:**
- Uses integers to represent time internally
- Neither of the functions for getting the preferred resolution nor for setting the resolution are supported
- Operation of this category is time-invariant: it does not use time to determine its output values or state updates
- Example: Time-invariant memory-less function (addition)

**Category 2:**
- Uses integers to represent time internally
- `getPreferredResolution` is supported, but `setResolution` is not supported
- FMU states which resolution it will use, but does not allow the master to change it
- The resolution is required not just preferred
- Example: FMU should output data at periodic time internals (e.g., periodic samplers or signal generators)

**Category 3:**
- Support `setResolution`, but do not support `getPreferredResolution`
- FMU is using the integer notation of time
- Any resolution is acceptable
- Example: Zero-Crossing Detector

**Category 4:**
- Supports both `getPreferredResolution` and `setResolution`
- FMU may first communicate to the master the resolution that it prefers
- Followed by the master telling the FMU what resolution it should use
- Example: ODE solver FMU

#### 3.3 Modular Support for FMI-HC
- Capabillity flags can be used to indicate which of the above categories an FMU belongs to
- Notifies the master about the capabilities of the FMU
- Master can adapt its behavior based on the capabilities of the FMU
- Basic Idea: Separation of concerns of the master algorithm from the logic that handles the translation between different resolution for different categories (done by wrappers around the FMUs)
- Wrapper: Software component that that translates function calls from the master algorithm to the FMUs
- Conversions between
    - Integer time resolutions
    - Integer time and floating-point time 

#### 3.3 Modular Support for FMI-HC
- Requires adopting the integer-based representation of time in the master algorithm and letting the master negotiate a time resolution as part of its initialization procedure
- Superdense time is modeled by allowing
    1. the master to take zero-size steps
    2. the simulation to ietarte over a number of lexicographically ordered indices before advancing to the next Newtonian time instant

**Example Implementation:** *FMI Integrated Development Environment (FIDE)*

1. Simulation tool reads a model description that describes how a set of FMUs is connected
2. Loads each FMU by reading the FMI XML-file and linking the required C-libraries
3. Each FMU is identified by a categry and a matching wrapper object is instantiated for every FMU
    - Wrapper is specifically designed to interface with FMUs of a particular category
    - All wrappers use integer time representation internally. Thus, the master algorithm can operate solely with integer time.
    - All necessary conversion between different resolutions and between integer time and floating-point time is handled by the wrappers

**During initialization:**

1. Master calls the function `determineResolution()` that iterates over all wrappers and queries them for the time resolution exponent using `getPreferredResolution()`
    - Time resolution exponent of the simulation is computed as the minimum among a default value and the exponents returned by the wrappers
2. Chosen resolution is communicated to all wrappers using `setResolution()`
    - Wrappers may convert integer time to the required model of time

- Environment must keep track of the global time of the master and the current step size using integers. Hence, it cannot use floating-point time internally.
- Function calls to FMUs are translated by the wrappers as needed
     

### 4. Time Conversion and Quantization

- Conversion reuqired when an FMU use a floating-point number internally for time keeping
- Conversion from floating-point represntation of time inside FMU to integer time in master comes with a precision loss
- Exampel: 
    1. Category 0b FMU rejects a proposed step size and makes partial progress over an interval of which the length cannot be losslesly converted into a corresponding fixed-resolution integer time
    2. Resolution mismatch between master and category 2 FMU
    3. Higher resolution integer time is converted to lower resolution integer time

**Key Insight**
- In co-simulation participants are treated as a black box (own understanding of time based on its local clock)
- Each level of hierarchy give rises to a different **clock domain** (passing of time may register differently from another clock domain)
- Degree of synchronization between two components depends on compatibility of their clock domains

**Problems**:
- Issue of translating time across different clock domains get complicated for corner cases
- May lead to Zeno behavior (infinite number of events in a finite time interval)
- May cause discrete events emitted by one component to be missed by another component
- Problems play only a role in interaction between category 0b and category 2 FMUs
    - Category 0b use floating-point time representation and require conversion from and to integer time
    - Category 2 FMUs cannot adapt to different resolutions

- Category 1 FMUs are time-invariant and do not pose any problems
- Category 3 and 4 FMUs can adapt to the resolution used by the master and hence do not pose any problems

#### 4.1 Converting from integer to real-valued time

- FMU internally represents time as a floating-point number
- Master / Ennvironment time resolution is given as a power of ten $r=10^{n}$ measured in seconds
- An integer time index $i$ scaled by a resolution $r$ can be converted to floating-point time $t$ as follows:
$$t = i \cdot r = i \cdot 10^{n} \, \mathrm{s}$$
- Step size for master and FMU are expressed separately, each in terms of their own loacl time representation
- **Master**: $i^\prime$ corresponds to the time $i^\prime \cdot r$
$$ i^\prime = i + \Delta i$$
- **FMU**: step size $\Delta t$ expressed in floating-point time and $t$ is the current time in the clock domain of the FMU
$$ t^\prime = t + \Delta t $$
- When **Master** and **FMU** agree on the next time step, we have:
$$ (i + \Delta i) \cdot r = t + \Delta t $$
- This gives for the time step of the FMU:
$$ \Delta t = (i + \Delta i) \cdot r - t $$

#### 4.2 Converting from real-valued to integer time

- Arbitrary real-valued time instant is unlikely to align perfectly with some integer-time instant
- Conversion from floating-point time to integer can introduce quantization errors up to one of the integer time resolution units
- The FMUs step size in the clock domain of the master should be:
$$ \Delta i = \frac{t + \Delta t}{r} - i $$
- Resulting value may not be an integer, possible solutions:
    1. Round down to the nearest integer (floor)
    2. Round up to the nearest integer (ceiling)
    3. Round to the nearest integer
- Choice: Ceiling function as it prevents quantization errors from blocking the progress of time
$$ \Delta i = \left \lceil \frac{t + \Delta t}{r} - i \right \rceil $$

#### 4.3 Converting between different-resolution integer times

- Conversions ate necessary for the support of fixed-resolution integer time FMUs (category 2)
- Master might operate in a finer resolution than the FMU
- Each time the master proposes a step size $\Delta i$ (expressed as a multiple of its own resolution $10^n$), this step must be converted into a step size $\Delta j$ expressed as a multiple of the FMU's resolution $10^k$
- Future time index of the FMU:
$$ j^\prime = j + \Delta j $$
- Target time index of the master:
$$ i^\prime = i + \Delta i $$
- After the successful step is completed, master and FMU are at the same time point. Consequently, 
$$i^\prime \cdot 10^n = j^\prime \cdot 10^k$$
- We obtain for the step size of the FMU:
$$ \Delta j = (i + \Delta i) \cdot 10^{n-k} - j $$
- The term $10^{n-k}$ can be a fraction when $n-k < 0$. This is possible because the master operates at a smaller or at the same resolution as the FMU
- Therefore, a quantization strategy is required:
    - Ceiling: Allows the FMU to move ahead of the master
    - Floor: FMU may lag behind the master
- Choice: Floor function since we quantize time for 0b FMUs such that they lag w.r.t. the master's clock
$$ \Delta j = \left \lfloor (i + \Delta i) \cdot 10^{n-k} \right \rfloor - j $$

- Conversion from a step in the clock domain of the FMU to a step in the clock domain of the master, we have:
$$ \Delta i =  (j + \Delta j) \cdot 10^{k-n} - i $$
- Because the master operates at a finer or at the same resolution as the FMU, the term $10^{k-n}$ is always an integer since $k-n \geq 0$

**Remarks:**
- Time quantization plays a different role for category 0b FMUs and category 2 FMUs
    - Category 0b: experience quantization only in the conversion from master time to FMU time
    - Category 2: experience quantization only in the conversion from FMU time to master time
    - Opposite directionality of quantization

**Example:**

![](figures/hybrid/hybrid_co_simulation_example_clock_domains.png)

- Two category 2 FMUs with different resolutions
- FMU A: resolution of $10^0 = 1 \,s$
- FMU B: resolution of $10^{1} = 10 \,s$
- Master uses resolution of FMU A ($1 \,s$)
- A step of size $1$ in the clock domain of FMU B reprsents a step of size $10$ in the clock domain of the master / FMU A
- Conversion the other way round from master to FMU B may not yield an integer and introduces quantization error
- The event generated by FMU A at internal time index $j_A = 15$ (corresponding to $15 \,s$) is converted to the time index of FMU B as follows:
$$ \Delta j_B = \left \lfloor (i + \Delta i) \cdot 10^{n-k} \right \rfloor - j_B$$
$$ \Delta j_B = \left \lfloor (0 + 15) \cdot 10^{0-1} \right \rfloor - 0 = \left \lfloor 1.5 \right \rfloor - 0 = 1 $$
- FMU B will see the event at its internal time index $j_B = 1$ (corresponding to $10 \,s$)
- This may look like a violation, but it is not, because the two internal clock domains are completely isolated from each other
- Can be compared to two people having a phone conversation, but one is looking at a clock that time is ahead compared to the clock of the other person
- Both cannot see each other's clocks and cannot compare them
- An outside observer (master) has its own clock
- In all three clock domains, causality is preserved.